# SmartFit LoRA full run: IDM-VTON on VITON-HD (spec §8.4, §23.9)

One pass over the 11,647 VITON-HD train pairs (updates of 16), about **15 GPU hours**: more than one Kaggle
session (12 h max). The same notebook runs every session: it continues from the newest checkpoint of the
previous session and, once training is complete, evaluates base vs LoRA on 48 held-out test pairs.

**Every session** (right-hand panel):
1. **Accelerator:** **GPU T4 x2**. **Internet:** On.
2. **Input → Add Input:** your private *SmartFit LoRA full pack* and your private *VITON-HD* dataset.
3. **Session 2 onwards, also:** Input → Add Input → **Your Work** → this notebook → its latest version's
   output (that is where the checkpoints are).
4. **Save Version → Save & Run All (Commit)**, so it keeps running with the browser closed.

Each session stops training at about 10 h 40 min, saves a checkpoint and zips the progress; nothing is lost.

In [ ]:
import glob, json, os, shutil, time

SESSION_START = time.time()
hits = [h for h in sorted(glob.glob("/kaggle/input/**/plan.json", recursive=True))
        if os.path.exists(os.path.join(os.path.dirname(h), "code", "train_lora.py"))]
assert hits, "Full-run pack not found: add the SmartFit LoRA full pack dataset as an input."
PACK = os.path.dirname(hits[0])
PLAN = json.load(open(f"{PACK}/plan.json"))
print("pack:", PACK)
print(json.dumps(PLAN, indent=2))

# VITON-HD root: the folder with train_pairs.txt whose train/image holds the full set
# (the 8-sample overfit pack also has a train_pairs.txt; it is ignored here).
def n_train_images(root):
    folder = os.path.join(root, "train", "image")
    return len(os.listdir(folder)) if os.path.isdir(folder) else 0

roots = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train_pairs.txt", recursive=True)},
               key=n_train_images, reverse=True)
assert roots and n_train_images(roots[0]) >= PLAN["train_samples"], (
    f"VITON-HD not found: add your private VITON-HD dataset (need {PLAN['train_samples']} train images; "
    f"found {[(r, n_train_images(r)) for r in roots]})")
VITON = roots[0]
records = [json.loads(l) for l in open(f"{PACK}/manifests/vitonhd_train.jsonl")]
fields = ("person_image_path", "cloth_image_path", "agnostic_mask_path", "densepose_path")
missing = [r[f] for r in records[:: max(1, len(records) // 300)] for f in fields
           if not os.path.exists(os.path.join(VITON, r[f]))]
assert not missing, f"VITON-HD files missing under {VITON}, e.g. {missing[:3]}"
print(f"VITON-HD: {VITON} ({n_train_images(VITON)} train images; manifest spot-check OK)")

# Continue from the previous session's output, if attached.
OUT = "/kaggle/working/lora_full"
os.makedirs(f"{OUT}/checkpoints", exist_ok=True)
step_of = lambda state: int(os.path.basename(os.path.dirname(state)).split("-")[1])
mine = glob.glob(f"{OUT}/checkpoints/step-*/state.pt")
prev = glob.glob("/kaggle/input/**/lora_full/checkpoints/step-*/state.pt", recursive=True)
if prev and (not mine or step_of(max(prev, key=step_of)) > step_of(max(mine, key=step_of))):
    newest = max(prev, key=step_of)
    src = os.path.dirname(newest)
    shutil.copytree(src, f"{OUT}/checkpoints/{os.path.basename(src)}", dirs_exist_ok=True)
    prev_out = os.path.dirname(os.path.dirname(src))
    if os.path.exists(f"{prev_out}/metrics.jsonl"):
        shutil.copy(f"{prev_out}/metrics.jsonl", f"{OUT}/metrics.jsonl")
    print(f"Continuing from step {step_of(newest)} of {PLAN['max_steps']} ({src})")
elif mine:
    print(f"Continuing from step {step_of(max(mine, key=step_of))} (this session's own checkpoint)")
else:
    print(f"Starting a new run: {PLAN['max_steps']} updates")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code at a fixed commit and an isolated environment

In [ ]:
![ -d /tmp/IDM-VTON ] || git clone -q https://github.com/yisol/IDM-VTON.git /tmp/IDM-VTON
!git -C /tmp/IDM-VTON checkout -q 0d5f3ec2d737487a9bb24e4100936ad254780383 && git -C /tmp/IDM-VTON log -1 --format="%H %cd"
!pip install -q uv==0.5.11
![ -x /tmp/env/bin/python ] || uv venv -q --python 3.11 /tmp/env
!uv pip install -q --python /tmp/env/bin/python torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q --python /tmp/env/bin/python -r {PACK}/code/requirements-lora.txt
!/tmp/env/bin/python -c "import torch, peft; print(torch.__version__, 'peft', peft.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"

## 2. Train (continues automatically; stops at the session's time budget with a checkpoint)
A line every 5 updates: loss, samples seen, seconds per update, peak memory per GPU.

In [ ]:
TRAIN_ARGS = " ".join([
    f"--repo /tmp/IDM-VTON --manifest {PACK}/manifests/vitonhd_train.jsonl --data-root {VITON} --out {OUT}",
    "--resume auto", f"--max-minutes {PLAN['session_train_minutes']}",
    f"--max-steps {PLAN['max_steps']} --grad-accum {PLAN['grad_accum']} --warmup-steps {PLAN['warmup_steps']}",
    f"--save-every {PLAN['save_every']} --log-every {PLAN['log_every']} --augment {PLAN['augment']} --num-workers 2",
])
!/tmp/env/bin/python {PACK}/code/train_lora.py {TRAIN_ARGS}

In [ ]:
import matplotlib.pyplot as plt

run = json.load(open(f"{OUT}/run.json"))
rows = [json.loads(l) for l in open(f"{OUT}/metrics.jsonl")]
FINISHED = run.get("status") == "completed"
print(json.dumps({k: run.get(k) for k in ("status", "final_step", "minutes", "peak_reserved_gb_by_gpu")}, indent=2))
print(f"progress: {run.get('final_step')} / {PLAN['max_steps']} updates")
if rows:
    print("seconds per update (median, this session):", sorted(r["sec_per_step"] for r in rows)[len(rows) // 2])
    plt.figure(figsize=(8, 3))
    plt.plot([r["step"] for r in rows], [r["loss"] for r in rows])
    plt.xlabel("update"); plt.ylabel("loss (mean of 5 updates)"); plt.title("LoRA full run"); plt.grid(alpha=.3)
    plt.show()
if not FINISHED:
    print("Not finished yet: run another session with this version's output attached as input.")

## 3. Held-out evaluation (only when training is complete)
48 VITON-HD test pairs (never trained on), base vs LoRA with the same seed and inputs. About 90 min.

In [ ]:
hours_left = 12 - (time.time() - SESSION_START) / 3600
if not FINISHED:
    print("Skipped: training is not complete yet.")
elif hours_left < 1.75:
    print(f"Skipped: only {hours_left:.1f} h left in this session. Run once more with this output attached; "
          "training is done, so the next session goes straight to evaluation.")
else:
    !/tmp/env/bin/python {PACK}/code/eval_lora.py --repo /tmp/IDM-VTON --manifest {PACK}/manifests/heldout_eval.jsonl --data-root {VITON} --adapter {OUT}/adapter --out {OUT}/eval

In [ ]:
from IPython.display import Image as Show, display

if os.path.exists(f"{OUT}/eval/results.json"):
    ev = json.load(open(f"{OUT}/eval/results.json"))
    print(json.dumps(ev["summary"], indent=2))
    display(Show(filename=f"{OUT}/eval/sheet.jpg", width=900))

## 4. Download
`results_lora_full.zip` (adapter, metrics, run summary, evaluation; no checkpoints). The checkpoints stay in this version's output for the next session. When the run is finished and evaluated, download the zip into the repo's `backend/` folder.

In [ ]:
!cd /kaggle/working && rm -f results_lora_full.zip && zip -qr results_lora_full.zip lora_full -x "lora_full/checkpoints/*" && ls -la /kaggle/working /kaggle/working/lora_full